# Track 4 — Embryo Dataset: dự đoán kết cục mang thai

Huấn luyện và đánh giá ba nhánh baseline của trên bộ **Embryo Dataset** (2.542 phôi, 1.415 cặp vợ chồng, nhãn `Nhan_MangThai` ∈ {0, 1}):

1. **Tabular** — Logistic Regression, Random Forest, LightGBM, XGBoost trên 36 đặc trưng
   lâm sàng/tinh dịch.
2. **Vision-only** — ResNet50 nhận ảnh phôi RGB (không có mặt nạ phân đoạn), fine-tune
   end-to-end.
3. **Multi-modal Late Fusion** — nối feature vector ảnh (2048-d) với feature vector bảng
   (16-d) rồi học chung qua MLP.

Toàn bộ ba nhánh dùng chung một khung đánh giá: `StratifiedGroupKFold` theo cặp vợ chồng
(`patient_id`), ngưỡng phân loại theo Youden's J, và Group Bootstrap 500 lần cho khoảng tin
cậy 95% của AUC-ROC.

Bộ dữ liệu có **hai cách chia** được dùng ở hai giai đoạn khác nhau trong notebook:
- **5-fold theo cặp** trên toàn bộ dữ liệu — dùng để khám phá/so sánh mô hình và đặc trưng
  (mục 4-6).
- **Bộ chia chính thức Cross-validation (2.056 phôi) / External-validation (486 phôi)** có
  sẵn trong `Data set splitting.zip` — dùng để chấm điểm cuối cùng, External chỉ chấm
  **đúng một lần**.

**Dữ liệu cần có** (Google Drive hoặc thư mục cục bộ, sửa đường dẫn ở mục 1):
```
Embryo dataset_2542.xlsx
Data set splitting/{Non-pregnant, Pregnant}/{Cross-validation, External-validation}/*.jpg
```

## 1. Cài đặt thư viện & cấu hình đường dẫn

In [ ]:
# Chỉ cần khi chạy trên Google Colab — bỏ qua nếu chạy local
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import os
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                              precision_recall_curve, confusion_matrix)

import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tvm

warnings.filterwarnings("ignore")

# ---- Cấu hình đường dẫn: SỬA cho đúng vị trí dữ liệu trên máy/Drive của bạn ----
EXCEL_PATH = r"/content/drive/MyDrive/Data/Embryo dataset_2542.xlsx"
SPLIT_DIR = Path(r"/content/drive/MyDrive/Data/Data set splitting")  # thư mục sau khi giải nén zip

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị tính toán: {DEVICE}")

## 2. Đọc dữ liệu & dịch tên cột

In [ ]:
column_mapping = {
    '胚胎': 'ID_Phoi', 'label': 'Nhan_MangThai', '女方年龄': 'Nu_Tuoi',
    '女方基础FSH': 'Nu_FSH', '女方基础PRL': 'Nu_PRL', '女方基础T': 'Nu_T',
    '女方基础P': 'Nu_P', '女方AMH': 'Nu_AMH', '女方白细胞计数': 'Nu_WBC_BachCau',
    '女方血小板': 'Nu_PLT_TieuCau', '女方血细胞压积': 'Nu_HCT', '女方凝血常规PT': 'Nu_PT',
    '女方凝血常规APPT': 'Nu_APTT', '女方凝血常规TT': 'Nu_TT', '女方D二聚体': 'Nu_D_Dimer',
    '女方甲功TPO': 'Nu_TPO', '女方空腹血糖': 'Nu_FBG_DuongHuyet', '女方肾功尿酸': 'Nu_UricAcid',
    '男方浓度': 'Nam_NongDo', '男方正常形态率': 'Nam_HinhThaiChuan', '男方PR': 'Nam_PR',
    '男方NP': 'Nam_NP', '男方存活率': 'Nam_TonHoai', '男方DFI': 'Nam_DFI',
    '男方ALT': 'Nam_ALT', '男方AST': 'Nam_AST', '男方血肌酐': 'Nam_Creatinine',
    '男方白细胞': 'Nam_WBC', '男方血小板': 'Nam_PLT', '男方空腹血糖': 'Nam_FBG',
    '女方用药Gn总天数': 'Nu_SoNgayDungGn', '女方用药HCG日P': 'Nu_HCG_P',
    '男方精液处理后NP': 'Nam_SauXuLy_NP', '男方精液处理方法': 'Nam_PhuongPhapXuLy',
    '男方精液处理后密度': 'Nam_SauXuLy_MatDo', '男方精液处理后PR': 'Nam_SauXuLy_PR',
    '男方精液处理后IM': 'Nam_SauXuLy_IM', '胚胎冷冻数': 'SoPhoiDongLanh',
}

df = pd.read_excel(EXCEL_PATH).rename(columns=column_mapping)

# Mã cặp vợ chồng/bệnh nhân suy từ ID_Phoi, ví dụ '1363_1' -> '1363'
df["patient_id"] = df["ID_Phoi"].astype(str).apply(lambda x: x.split("_")[0] if "_" in x else x)

ID_COL, LABEL_COL, GROUP_COL = "ID_Phoi", "Nhan_MangThai", "patient_id"

FEATURES = [c for c in df.columns if c not in (ID_COL, LABEL_COL, GROUP_COL)]
non_numeric = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
if non_numeric:
    print("Cột không phải số (tạm bỏ khỏi FEATURES):", non_numeric)
    FEATURES = [c for c in FEATURES if c not in non_numeric]

X = df[FEATURES].copy()
y = df[LABEL_COL].astype(int).values
groups = df[GROUP_COL].astype(str).values

print(f"Tổng số mẫu phôi: {df.shape[0]} | Số cặp vợ chồng: {df[GROUP_COL].nunique()}")
print(f"Số đặc trưng: {len(FEATURES)} | Số giá trị thiếu: {int(X.isna().sum().sum())}")
print(f"Tỷ lệ đậu thai: {y.mean():.1%} (đoán mò 'không đậu' đúng {1 - y.mean():.1%})")

## 3. Vì sao phải chia tập theo cặp vợ chồng, không theo dòng?

Mỗi cặp vợ chồng có thể có nhiều phôi, và các phôi cùng cặp dùng chung hồ sơ lâm sàng
(tuổi, AMH, chỉ số tinh dịch...). Nếu chia ngẫu nhiên theo dòng, hai phôi của cùng một cặp có
thể rơi vào cả train lẫn test — mô hình "nhìn trước" hồ sơ của test qua train, AUC bị thổi
phồng. Ba ô dưới đây kiểm chứng điều này trước khi xây pipeline chính thức.

In [ ]:
# Mỗi cặp có bao nhiêu phôi?
per_couple = df.groupby(GROUP_COL).size()
print("Số phôi / cặp:\n", per_couple.value_counts().sort_index())

# Đặc trưng lâm sàng có giống hệt nhau giữa các phôi cùng cặp không?
same = df.groupby(GROUP_COL)[FEATURES].nunique(dropna=False).le(1).mean()
print(f"\nTỷ lệ cặp có đặc trưng đồng nhất giữa các phôi: {same.mean():.1%} (min {same.min():.1%})")

# Nhãn có giống nhau giữa các phôi cùng cặp không?
multi = df.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
multi = multi[multi["size"] > 1]
print(f"Cặp có ≥2 phôi: {len(multi)} | trong đó nhãn giống hệt nhau: {(multi['nunique'] == 1).mean():.1%}")

In [ ]:
def cv_auc(splitter, make_model):
    oof = np.zeros(len(y))
    for tr, te in splitter:
        m = make_model().fit(X.iloc[tr], y[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y, oof)

mk_lgb = lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                     min_child_samples=5, random_state=42, verbose=-1)
mk_rf = lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

rows = []
for name, mk in [("LightGBM", mk_lgb), ("RandomForest", mk_rf)]:
    a_row = cv_auc(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y), mk)
    a_grp = cv_auc(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, y, groups), mk)
    rows.append((name, a_row, a_grp, a_row - a_grp))

leak_tbl = pd.DataFrame(
    rows, columns=["Model", "AUC chia theo DÒNG (sai)", "AUC chia theo CẶP (đúng)", "Chênh lệch"]
).round(3)
print(leak_tbl.to_string(index=False))

# % dòng test có "bản sao" hồ sơ lâm sàng y hệt trong train, khi chia theo dòng
fingerprint = pd.factorize(df[FEATURES].round(6).astype(str).agg("|".join, axis=1))[0]
fp_s = pd.Series(fingerprint)
dup = np.mean([fp_s.iloc[te].isin(set(fp_s.iloc[tr])).mean()
               for tr, te in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y)])
print(f"\nChia theo dòng: {dup:.0%} dòng test có hồ sơ y hệt trong train -> phải chia theo cặp.")

## 4. Hàm tiện ích dùng chung: ngưỡng Youden & Group Bootstrap CI

Dùng lại cho **cả ba nhánh** (Tabular, Vision/Fusion) để đảm bảo mọi nhánh được chấm điểm theo cùng một quy ước.

In [ ]:
def youden_threshold(y_true, y_prob):
    """Ngưỡng phân loại tối ưu theo chỉ số Youden's J (TPR - FPR)."""
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return thr[np.argmax(tpr - fpr)]

def group_bootstrap_ci(y_true, y_prob, groups, n_boot=500, seed=0):
    """Khoảng tin cậy 95% cho AUC-ROC, bootstrap theo cặp vợ chồng (không theo từng phôi)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    idx_by_group = {g: np.where(groups == g)[0] for g in unique_groups}
    scores = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, len(unique_groups), replace=True)
        idx = np.concatenate([idx_by_group[g] for g in sampled_groups])
        if len(np.unique(y_true[idx])) == 2:
            scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(scores, [2.5, 97.5])

def compute_5_metrics(y_true, y_prob, threshold=None):
    """AUC-ROC, AUC-PR, Precision, Sensitivity, Specificity tại một ngưỡng cho trước
    (hoặc ngưỡng Youden nếu không truyền vào)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if threshold is None:
        threshold = youden_threshold(y_true, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp_, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "AUC-ROC": roc_auc_score(y_true, y_prob),
        "AUC-PR": average_precision_score(y_true, y_prob),
        "Precision": tp / max(tp + fp_, 1),
        "Sensitivity": tp / max(tp + fn, 1),
        "Specificity": tn / max(tn + fp_, 1),
        "Threshold": threshold,
    }

## 5. Nhánh Tabular: nested `StratifiedGroupKFold` (5-fold, toàn bộ dữ liệu)

Vòng ngoài (5 fold) chấm điểm trên test; vòng trong (8 fold) tách thêm validation để chọn
ngưỡng Youden — tránh chọn ngưỡng trên chính tập được chấm điểm. Dùng để khám phá/so sánh mô
hình trước khi chuyển sang bộ chia chính thức.

In [ ]:
def make_models(seed=42):
    models = {
        "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=500, min_samples_leaf=3, n_jobs=-1, random_state=seed),
        "LightGBM": lambda: lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.03, num_leaves=8, min_child_samples=20,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
            importance_type="gain", random_state=seed, verbose=-1),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8,
            colsample_bytree=0.8, eval_metric="auc", random_state=seed, verbosity=0)
    return models

def group_cv(X, y, groups, n_splits=5, seed=42):
    models = make_models(seed)
    n = len(y)
    oof_p = {k: np.zeros(n) for k in models}
    oof_c = {k: np.zeros(n, dtype=int) for k in models}
    fold_auc = {k: [] for k in models}
    importance = []

    outer = StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed)
    for k, (tr, te) in enumerate(outer.split(X, y, groups)):
        inner = StratifiedGroupKFold(8, shuffle=True, random_state=seed + k)
        i_tr, i_va = next(inner.split(X.iloc[tr], y[tr], groups[tr]))
        tr_i, va_i = tr[i_tr], tr[i_va]

        assert not (set(groups[tr_i]) & set(groups[va_i])), "LEAK train/val"
        assert not (set(groups[tr_i]) & set(groups[te])), "LEAK train/test"
        assert not (set(groups[va_i]) & set(groups[te])), "LEAK val/test"

        print(f"fold {k}: train={len(tr_i)} val={len(va_i)} test={len(te)}")
        for name, mk in models.items():
            m = mk().fit(X.iloc[tr_i], y[tr_i])
            thr = youden_threshold(y[va_i], m.predict_proba(X.iloc[va_i])[:, 1])  # ngưỡng chọn trên VAL
            p = m.predict_proba(X.iloc[te])[:, 1]
            oof_p[name][te] = p
            oof_c[name][te] = (p >= thr).astype(int)
            fold_auc[name].append(roc_auc_score(y[te], p))
            if name == "LightGBM":
                importance.append(pd.Series(m.feature_importances_, index=X.columns))
    return oof_p, oof_c, fold_auc, importance

def summarize(y, groups, oof_p, oof_c, fold_auc=None):
    rows = []
    for name in oof_p:
        pred, p = oof_c[name], oof_p[name]
        tp = ((pred == 1) & (y == 1)).sum(); fp_ = ((pred == 1) & (y == 0)).sum()
        tn = ((pred == 0) & (y == 0)).sum(); fn = ((pred == 0) & (y == 1)).sum()
        lo, hi = group_bootstrap_ci(y, p, groups)
        row = {
            "Model": name,
            "AUC-ROC": roc_auc_score(y, p),
            "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "AUC-PR": average_precision_score(y, p),
            "Precision": tp / max(tp + fp_, 1),
            "Sensitivity": tp / max(tp + fn, 1),
            "Specificity": tn / max(tn + fp_, 1),
        }
        if fold_auc:
            row["AUC/fold"] = f"{np.mean(fold_auc[name]):.3f}±{np.std(fold_auc[name]):.3f}"
        rows.append(row)
    return pd.DataFrame(rows).round(3)

oof_p, oof_c, fold_auc, importance = group_cv(X, y, groups)
res_tab = summarize(y, groups, oof_p, oof_c, fold_auc)
print(f"\n(AUC-PR của đoán mò = tỷ lệ dương = {y.mean():.3f})")
res_tab

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for n, p in oof_p.items():
    fpr, tpr, _ = roc_curve(y, p)
    ax[0].plot(fpr, tpr, label=f"{n} ({roc_auc_score(y, p):.3f})")
    pr, rc, _ = precision_recall_curve(y, p)
    ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set(title="ROC (chia theo cặp)", xlabel="1 - Specificity", ylabel="Sensitivity")
ax[1].axhline(y.mean(), color="k", ls="--", lw=0.8)
ax[1].set(title="Precision-Recall", xlabel="Recall", ylabel="Precision")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
imp = pd.concat(importance, axis=1).mean(axis=1).sort_values(ascending=False)
imp.head(15).iloc[::-1].plot.barh(figsize=(7, 5), title="LightGBM – độ quan trọng (gain, trung bình 5 fold)")
plt.tight_layout(); plt.show()

if "SoPhoiDongLanh" in X.columns:
    X_ablation = X.drop(columns=["SoPhoiDongLanh"])
    p2, c2, f2, _ = group_cv(X_ablation, y, groups)
    print("\n=== KHÔNG có SoPhoiDongLanh ===")
    print(summarize(y, groups, p2, c2, f2).to_string(index=False))

## 6. Đọc bộ chia chính thức Cross-validation / External-validation

Bộ dữ liệu gốc đã chia sẵn **Cross-validation** (2.056 phôi) và **External-validation**
(486 phôi) theo thư mục `Data set splitting/{Non-pregnant, Pregnant}/{Cross-validation,
External-validation}`. Ô dưới đây quét toàn bộ ảnh, ghép `ID_Phoi` suy từ tên file, và kiểm
tra không có cặp vợ chồng nào xuất hiện ở cả hai tập.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
files = [p for p in SPLIT_DIR.rglob("*") if p.suffix.lower() in IMG_EXT]
print(f"Tìm thấy {len(files)} ảnh")
for folder, count in sorted(Counter(str(p.parent.relative_to(SPLIT_DIR)) for p in files).items()):
    print(f"  {count:5d}  {folder}")

ids = set(df[ID_COL].astype(str))

def find_id(stem):
    if stem in ids:
        return stem
    for m in re.findall(r"\d+_\d+", stem):
        if m in ids:
            return m
    return None

def guess_split(parts):
    s = " ".join(parts).lower()
    if "external" in s:
        return "external"
    if "cross" in s or "train" in s:
        return "cv"
    return "unknown"

def guess_label(parts):
    s = " ".join(parts).lower()
    if re.search(r"non[\s_-]*pregnan", s):
        return 0
    if "pregnan" in s:
        return 1
    return np.nan

manifest = pd.DataFrame([{
    "path": str(p), "ID_Phoi": find_id(p.stem),
    "split": guess_split(p.relative_to(SPLIT_DIR).parts[:-1]),
    "label_folder": guess_label(p.relative_to(SPLIT_DIR).parts[:-1]),
} for p in files])

print(f"\nGhép được ID_Phoi cho {manifest['ID_Phoi'].notna().mean():.1%} ảnh")
m = manifest.dropna(subset=["ID_Phoi"]).merge(df, left_on="ID_Phoi", right_on=ID_COL)
print(f"Nhãn thư mục khớp {LABEL_COL}: {(m['label_folder'] == m[LABEL_COL]).mean():.1%}")
print("\nSố ảnh theo tập:\n", m["split"].value_counts())
overlap = set(m.loc[m.split == "cv", GROUP_COL]) & set(m.loc[m.split == "external", GROUP_COL])
print(f"Số cặp vợ chồng xuất hiện ở CẢ hai tập: {len(overlap)} (mong muốn = 0)")

## 7. Đánh giá Tabular cuối trên bộ chia chính thức (External chỉ chấm một lần)

Quy trình chuẩn: 5-fold theo cặp trên tập **Cross-validation** để chọn mô hình/ngưỡng →
huấn luyện lại trên **toàn bộ** tập CV → chấm điểm **đúng một lần** trên **External**.

In [ ]:
cv_mask = df[ID_COL].astype(str).isin(set(m.loc[m.split == "cv", "ID_Phoi"])).values
ext_mask = df[ID_COL].astype(str).isin(set(m.loc[m.split == "external", "ID_Phoi"])).values
print(f"CV: {cv_mask.sum()} phôi | External: {ext_mask.sum()} phôi")

X_cv, y_cv, g_cv = X[cv_mask], y[cv_mask], groups[cv_mask]
X_ex, y_ex, g_ex = X[ext_mask], y[ext_mask], groups[ext_mask]

p_cv, c_cv, f_cv, _ = group_cv(X_cv, y_cv, g_cv)
print("\n=== 5-fold theo cặp trên tập Cross-validation ===")
print(summarize(y_cv, g_cv, p_cv, c_cv, f_cv).to_string(index=False))

p_ex, c_ex = {}, {}
for name, mk in make_models().items():
    thr = youden_threshold(y_cv, p_cv[name])
    mdl = mk().fit(X_cv, y_cv)
    p_ex[name] = mdl.predict_proba(X_ex)[:, 1]
    c_ex[name] = (p_ex[name] >= thr).astype(int)

print("\n=== EXTERNAL VALIDATION (số liệu Tabular chính thức để báo cáo) ===")
ext_tab = summarize(y_ex, g_ex, p_ex, c_ex)
print(ext_tab.to_string(index=False))

## 8. Vision-only & Multi-modal Late Fusion

- **Vision-only**: ResNet50 nhận ảnh RGB chuẩn 3 kênh (Embryo Dataset không có mặt nạ phân
  đoạn như bộ Master/PCRM), fine-tune end-to-end.
- **Late Fusion**: cùng backbone ResNet50 (bỏ lớp `fc`, giữ vector 2048-d) nối với một Tabular
  Encoder nhỏ (36 đặc trưng lâm sàng → 16-d) rồi qua MLP phân loại — đúng định nghĩa "nối
  feature vector", không phải lấy trung bình xác suất hai mô hình riêng.
- Quy trình đánh giá giống hệt mục 8: 5-fold trên CV để chọn ngưỡng → huấn luyện lại trên
  toàn bộ CV → chấm một lần trên External.

In [ ]:
img_df = m.copy()  # path, ID_Phoi, split, label_folder, + toàn bộ cột của df (FEATURES, GROUP_COL,...)
df_cv_img = img_df[img_df.split == "cv"].reset_index(drop=True)
df_ext_img = img_df[img_df.split == "external"].reset_index(drop=True)
print(f"Ảnh khớp đủ Tabular+Ảnh -> CV: {len(df_cv_img)} | External: {len(df_ext_img)}")

y_cv_img = df_cv_img[LABEL_COL].astype(int).values
y_ext_img = df_ext_img[LABEL_COL].astype(int).values
groups_cv_img = df_cv_img[GROUP_COL].astype(str).values
groups_ext_img = df_ext_img[GROUP_COL].astype(str).values

# ---- Transform ảnh: 224x224 + ImageNet mean/std + augmentation trên train ----
train_transform = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.5),
    T.RandomRotation(15), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class EmbryoMultiModalDataset(Dataset):
    """Trả về (ảnh RGB 3 kênh, vector Tabular đã impute, nhãn)."""

    def __init__(self, df_img, tab_matrix, transform):
        self.paths = df_img["path"].values
        self.tab = np.asarray(tab_matrix, dtype=np.float32)
        self.y = df_img[LABEL_COL].astype(np.float32).values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=0)
        img = self.transform(img)
        tab = torch.tensor(self.tab[idx], dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.float32)
        return img, tab, label


def impute_median_foldwise(train_df, other_df, cols):
    """Impute median CHỈ học trên phần train của từng fold, tránh rò rỉ từ val/test."""
    med = train_df[cols].median()
    return (train_df[cols].fillna(med).values.astype(np.float32),
            other_df[cols].fillna(med).values.astype(np.float32), med)


class VisionOnlyResNet50(nn.Module):
    """Baseline Vision-only — ảnh RGB chuẩn 3 kênh."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = tvm.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = tvm.resnet50(weights=weights)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_ftrs, 1))

    def forward(self, x):
        return self.backbone(x).squeeze(-1)


class LateFusionResNet50(nn.Module):
    """Late Fusion: nối feature vector Vision (2048-d) với Tabular Encoder (16-d) rồi
    học chung qua MLP end-to-end."""

    def __init__(self, num_tabular_features, pretrained=True):
        super().__init__()
        weights = tvm.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = tvm.resnet50(weights=weights)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.tab_encoder = nn.Sequential(
            nn.Linear(num_tabular_features, 16), nn.BatchNorm1d(16), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(num_ftrs + 16, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, img_x, tab_x):
        v = self.backbone(img_x)
        t = self.tab_encoder(tab_x)
        return self.classifier(torch.cat([v, t], dim=1)).squeeze(-1)


N_SPLITS_IMG = 5
EPOCHS_IMG = 6 
BATCH_SIZE_IMG = 32
LR_IMG = 1e-4
criterion_img = nn.BCEWithLogitsLoss()
print("Đã thiết lập xong Dataset / kiến trúc / cấu hình huấn luyện cho Vision & Fusion.")

In [ ]:
def train_image_branch(model_ctor, is_fusion, label):
    """5-fold CV trên tập Cross-validation, trả về dự đoán out-of-fold."""
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS_IMG, shuffle=True, random_state=42)
    oof = np.zeros(len(df_cv_img))

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(df_cv_img, y_cv_img, groups_cv_img)):
        tr_df, va_df = df_cv_img.iloc[tr_idx], df_cv_img.iloc[va_idx]
        X_tr_imp, X_va_imp, _ = impute_median_foldwise(tr_df, va_df, FEATURES)

        train_loader = DataLoader(EmbryoMultiModalDataset(tr_df, X_tr_imp, train_transform),
                                   batch_size=BATCH_SIZE_IMG, shuffle=True, num_workers=2)
        val_loader = DataLoader(EmbryoMultiModalDataset(va_df, X_va_imp, eval_transform),
                                 batch_size=BATCH_SIZE_IMG, shuffle=False, num_workers=2)

        model = model_ctor().to(DEVICE)
        opt = optim.AdamW(model.parameters(), lr=LR_IMG, weight_decay=1e-2)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_IMG)

        for _ in range(EPOCHS_IMG):
            model.train()
            for imgs, tabs, labels in train_loader:
                imgs, tabs, labels = imgs.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
                opt.zero_grad()
                out = model(imgs, tabs) if is_fusion else model(imgs)
                loss = criterion_img(out, labels)
                loss.backward(); opt.step()
            sch.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for imgs, tabs, _ in val_loader:
                imgs, tabs = imgs.to(DEVICE), tabs.to(DEVICE)
                out = model(imgs, tabs) if is_fusion else model(imgs)
                preds.extend(torch.sigmoid(out).cpu().numpy())
        oof[va_idx] = preds
        print(f"  Fold {fold + 1} {label} AUC-ROC: {roc_auc_score(y_cv_img[va_idx], preds):.3f}")

    return oof


def train_full_and_score_external(model_ctor, is_fusion, label):
    """Huấn luyện lại trên toàn bộ tập CV, chấm đúng một lần trên External."""
    med_full = df_cv_img[FEATURES].median()
    X_full_imp = df_cv_img[FEATURES].fillna(med_full).values.astype(np.float32)
    X_ext_imp = df_ext_img[FEATURES].fillna(med_full).values.astype(np.float32)

    full_loader = DataLoader(EmbryoMultiModalDataset(df_cv_img, X_full_imp, train_transform),
                              batch_size=BATCH_SIZE_IMG, shuffle=True, num_workers=2)
    ext_loader = DataLoader(EmbryoMultiModalDataset(df_ext_img, X_ext_imp, eval_transform),
                             batch_size=BATCH_SIZE_IMG, shuffle=False)

    model = model_ctor().to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=LR_IMG, weight_decay=1e-2)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_IMG)

    for _ in range(EPOCHS_IMG):
        model.train()
        for imgs, tabs, labels in full_loader:
            imgs, tabs, labels = imgs.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            out = model(imgs, tabs) if is_fusion else model(imgs)
            loss = criterion_img(out, labels)
            loss.backward(); opt.step()
        sch.step()

    model.eval()
    preds = []
    with torch.no_grad():
        for imgs, tabs, _ in ext_loader:
            imgs, tabs = imgs.to(DEVICE), tabs.to(DEVICE)
            out = model(imgs, tabs) if is_fusion else model(imgs)
            preds.extend(torch.sigmoid(out).cpu().numpy())
    print(f"  External {label} AUC-ROC: {roc_auc_score(y_ext_img, preds):.3f}")
    return np.array(preds)


print("=== [1/2] VISION-ONLY (ResNet50) ===")
oof_vis = train_image_branch(lambda: VisionOnlyResNet50(pretrained=True), is_fusion=False, label="Vision")
thr_vis = youden_threshold(y_cv_img, oof_vis)
ext_vis = train_full_and_score_external(lambda: VisionOnlyResNet50(pretrained=True), is_fusion=False, label="Vision")

print("\n=== [2/2] MULTI-MODAL LATE FUSION ===")
oof_fus = train_image_branch(lambda: LateFusionResNet50(len(FEATURES), pretrained=True), is_fusion=True, label="Fusion")
thr_fus = youden_threshold(y_cv_img, oof_fus)
ext_fus = train_full_and_score_external(lambda: LateFusionResNet50(len(FEATURES), pretrained=True), is_fusion=True, label="Fusion")

## 10. Bảng tổng hợp cuối cùng

Gộp Tabular (mục 8), Vision-only và Fusion (mục 9) vào một bảng duy nhất, mỗi nhánh có cả
`Cross-Validation (OOF)` lẫn `External-Validation`, cùng cấu trúc cột để đối chiếu trực tiếp
với bảng trong báo cáo kỹ thuật.

In [ ]:
rows = []

for name in make_models():
    rows.append({"Dataset": "Embryo Data", "Branch": f"Tabular – {name}",
                 "Split": "Cross-Validation (OOF)",
                 **{k: v for k, v in summarize(y_cv, g_cv, p_cv, c_cv, f_cv)
                    .set_index("Model").loc[name, ["AUC-ROC", "95% CI", "AUC-PR",
                                                    "Precision", "Sensitivity", "Specificity"]].items()}})
    rows.append({"Dataset": "Embryo Data", "Branch": f"Tabular – {name}",
                 "Split": "External-Validation",
                 **{k: v for k, v in ext_tab.set_index("Model").loc[name, ["AUC-ROC", "95% CI", "AUC-PR",
                                                    "Precision", "Sensitivity", "Specificity"]].items()}})

branch_specs = [
    ("Vision-only (ResNet50)", oof_vis, thr_vis, ext_vis, y_cv_img, groups_cv_img, y_ext_img, groups_ext_img),
    ("Multi-modal Fusion (ResNet50+MLP)", oof_fus, thr_fus, ext_fus, y_cv_img, groups_cv_img, y_ext_img, groups_ext_img),
]
for name, oof, thr, ext, y_cv_, g_cv_, y_ext_, g_ext_ in branch_specs:
    m_cv = compute_5_metrics(y_cv_, oof, thr); lo, hi = group_bootstrap_ci(y_cv_, oof, g_cv_)
    rows.append({"Dataset": "Embryo Data", "Branch": name, "Split": "Cross-Validation (OOF)",
                 "AUC-ROC": round(m_cv["AUC-ROC"], 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                 "AUC-PR": round(m_cv["AUC-PR"], 3), "Precision": round(m_cv["Precision"], 3),
                 "Sensitivity": round(m_cv["Sensitivity"], 3), "Specificity": round(m_cv["Specificity"], 3)})
    m_ex = compute_5_metrics(y_ext_, ext, thr); lo, hi = group_bootstrap_ci(y_ext_, ext, g_ext_)
    rows.append({"Dataset": "Embryo Data", "Branch": name, "Split": "External-Validation",
                 "AUC-ROC": round(m_ex["AUC-ROC"], 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                 "AUC-PR": round(m_ex["AUC-PR"], 3), "Precision": round(m_ex["Precision"], 3),
                 "Sensitivity": round(m_ex["Sensitivity"], 3), "Specificity": round(m_ex["Specificity"], 3)})

final_summary = pd.DataFrame(rows)[
    ["Dataset", "Branch", "Split", "AUC-ROC", "95% CI", "AUC-PR", "Precision", "Sensitivity", "Specificity"]]

print("=" * 90)
print(" BẢNG TỔNG HỢP TRACK 4 — EMBRYO DATA (Tabular + Vision + Fusion, CV & External) ")
print("=" * 90)
final_summary.to_csv("track4_embryo_final_summary.csv", index=False)
final_summary